<a href="https://colab.research.google.com/github/KP-365/Skinrash-detection/blob/main/BugBitesAug2model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from datasets import load_dataset
import numpy as np

SEED = 1337
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 8
BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS_HEAD = 150
EPOCHS_FINETUNE = 75
LR_HEAD = 1e-3
LR_FINETUNE = 3e-5
DROPOUT_P = 0.4
PATIENCE = 7

print(f"Using device: {DEVICE}")

Using device: cuda


In [20]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

In [21]:
clean = load_dataset("eceunal/bug-bite-images-hf")
labels = clean["train"].features["label"].names
print("Classes:", labels)
print(f"Train (raw, pre-augmentation): {len(clean['train'])}")
print(f"Validation (raw, untouched):   {len(clean['validation'])}")
print(f"Test (raw, untouched):         {len(clean['test'])}")

Classes: ['ants', 'bed_bugs', 'chiggers', 'fleas', 'mosquitos', 'no_bites', 'spiders', 'ticks']
Train (raw, pre-augmentation): 896
Validation (raw, untouched):   106
Test (raw, untouched):         53


In [22]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(30),
    transforms.RandomAffine(degrees=15, translate=(0.15, 0.15), scale=(0.8, 1.2), shear=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [23]:
class BugBiteDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        img = ex["image"].convert("RGB")
        img = self.transform(img)
        return img, ex["label"]

train_ds = BugBiteDataset(clean["train"], train_transform)
val_ds = BugBiteDataset(clean["validation"], eval_transform)
test_ds = BugBiteDataset(clean["test"], eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

Train batches: 28, Val batches: 4, Test batches: 2


In [24]:
def build_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=DROPOUT_P),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

model = build_model()
print(model.classifier)

Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=1280, out_features=8, bias=True)
)


In [25]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, targets in loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

criterion = nn.CrossEntropyLoss()

In [26]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
early_stopper = EarlyStopping(patience=PATIENCE)

print("\n=== Phase 1: Head-only training (frozen backbone) ===")
best_val_acc = 0
best_epoch_phase1 = 0
for epoch in range(EPOCHS_HEAD):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS_HEAD} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch_phase1 = epoch + 1
        torch.save(model.state_dict(), "best_model_aggaug_phase1.pt")
        print(f"  -> New best (epoch {epoch+1}), checkpoint saved.")

    early_stopper(val_loss)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1} (phase 1)")
        break

print(f"\nPhase 1 complete. Best val_acc={best_val_acc:.4f} at epoch {best_epoch_phase1}")


=== Phase 1: Head-only training (frozen backbone) ===
Epoch 1/150 | train_loss=2.0041 train_acc=0.2132 | val_loss=1.9072 val_acc=0.3113
  -> New best (epoch 1), checkpoint saved.
Epoch 2/150 | train_loss=1.7530 train_acc=0.4085 | val_loss=1.7859 val_acc=0.3774
  -> New best (epoch 2), checkpoint saved.
Epoch 3/150 | train_loss=1.6296 train_acc=0.4509 | val_loss=1.7377 val_acc=0.4057
  -> New best (epoch 3), checkpoint saved.
Epoch 4/150 | train_loss=1.5317 train_acc=0.4866 | val_loss=1.6853 val_acc=0.4057
Epoch 5/150 | train_loss=1.4686 train_acc=0.4978 | val_loss=1.6445 val_acc=0.4151
  -> New best (epoch 5), checkpoint saved.
Epoch 6/150 | train_loss=1.4425 train_acc=0.5078 | val_loss=1.6175 val_acc=0.4151
Epoch 7/150 | train_loss=1.4030 train_acc=0.5011 | val_loss=1.5883 val_acc=0.4434
  -> New best (epoch 7), checkpoint saved.
Epoch 8/150 | train_loss=1.3966 train_acc=0.5100 | val_loss=1.5843 val_acc=0.4057
Epoch 9/150 | train_loss=1.3684 train_acc=0.5290 | val_loss=1.5384 val_acc

In [27]:
print("\n=== Phase 2: Fine-tuning last 3 blocks ===")
model.load_state_dict(torch.load("best_model_aggaug_phase1.pt"))

for param in model.features[-3:].parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
early_stopper = EarlyStopping(patience=PATIENCE)

best_val_acc = 0
best_epoch_phase2 = 0
for epoch in range(EPOCHS_FINETUNE):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS_FINETUNE} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch_phase2 = epoch + 1
        torch.save(model.state_dict(), "best_model_aggaug_final.pt")
        print(f"  -> New best (epoch {epoch+1}), checkpoint saved.")

    early_stopper(val_loss)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1} (phase 2)")
        break

print(f"\nPhase 2 complete. Best val_acc={best_val_acc:.4f} at epoch {best_epoch_phase2}")


=== Phase 2: Fine-tuning last 3 blocks ===
Epoch 1/75 | train_loss=1.2024 train_acc=0.5725 | val_loss=1.3681 val_acc=0.5189
  -> New best (epoch 1), checkpoint saved.
Epoch 2/75 | train_loss=1.1199 train_acc=0.6004 | val_loss=1.3418 val_acc=0.5000
Epoch 3/75 | train_loss=1.0864 train_acc=0.6183 | val_loss=1.3322 val_acc=0.5094
Epoch 4/75 | train_loss=1.0449 train_acc=0.6373 | val_loss=1.3226 val_acc=0.5377
  -> New best (epoch 4), checkpoint saved.
Epoch 5/75 | train_loss=1.0471 train_acc=0.6150 | val_loss=1.3000 val_acc=0.5377
Epoch 6/75 | train_loss=0.9933 train_acc=0.6562 | val_loss=1.2995 val_acc=0.5283
Epoch 7/75 | train_loss=1.0133 train_acc=0.6429 | val_loss=1.2745 val_acc=0.5377
Epoch 8/75 | train_loss=0.9842 train_acc=0.6529 | val_loss=1.2746 val_acc=0.5660
  -> New best (epoch 8), checkpoint saved.
Epoch 9/75 | train_loss=1.0039 train_acc=0.6350 | val_loss=1.2589 val_acc=0.5472
Epoch 10/75 | train_loss=0.9217 train_acc=0.6808 | val_loss=1.2526 val_acc=0.5377
Epoch 11/75 | tr

In [28]:
model.load_state_dict(torch.load("best_model_aggaug_final.pt"))
test_loss, test_acc = eval_epoch(model, test_loader, criterion)
print(f"\nFinal TEST accuracy: {test_acc:.4f} | test_loss: {test_loss:.4f}")


Final TEST accuracy: 0.6792 | test_loss: 0.8368


In [32]:
from google.colab import files
files.download("best_model_aggaug_final.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>